In [49]:
import torch
import csv
from torch_geometric.data import Data
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv

In [59]:
node_features = []
edge_attr = []

with open("data_clean.csv", "r") as file:
    csv_reader = csv.DictReader(file)

    for row in csv_reader:
        to_lat = float(row["to_lat"])
        to_lon = float(row["to_lon"])
        node_features.append([to_lat, to_lon])
        
        duration = float(row["predicted_time_min"])
        edge_attr.append([duration])

print(node_features)

[[47.217621, 39.707611], [47.221096, 39.69097], [47.225482, 39.717826], [47.255549, 39.644401], [47.224904, 39.711349], [47.210328, 39.692724], [47.233896, 39.61461], [47.223208, 39.697369], [47.218795, 39.707842], [47.252899, 39.764581], [47.214927, 39.705652], [47.273902, 39.684949], [47.226024, 39.729858], [47.229064, 39.628654], [47.219161, 39.705761], [47.249401, 39.735175], [47.245568, 39.680052], [47.219812, 39.715458], [47.238981, 39.694731], [47.22192, 39.724959], [47.230965, 39.686521], [47.226262, 39.73101], [47.256223, 39.704661], [47.222052, 39.715387], [47.297733, 39.701845], [47.296659, 39.717094], [47.21889, 39.713986], [47.223083, 39.698421], [47.222549, 39.72163], [47.231065, 39.75563], [47.284506, 39.720996], [47.208051, 39.634353], [47.252037, 39.795119], [47.2102, 39.693253], [47.252637, 39.713811], [47.214527, 39.618866], [47.25344, 39.70265], [47.23422, 39.601897], [47.157576, 39.548447], [47.225003, 39.701557], [47.246689, 39.768088], [47.192543, 39.619834], [47

In [55]:
# NODES \ INPUT PARAMETRS
node_features = torch.tensor(node_features, dtype=torch.float)

# CONNECTION BETWEEN NODES
edge_index = torch.combinations(torch.arange(len(node_features)), r = 2).t()
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim = 1)

# PARAMETRS OF CONNECTIONS
edge_attr = torch.tensor(edge_attr, dtype=torch.float)

data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_attr)

print(edge_index)

tensor([[ 0,  0,  0,  ..., 98, 99, 99],
        [ 1,  2,  3,  ..., 97, 97, 98]])


C:\Users\vladulik\AppData\Local\Temp\ipykernel_3024\1643927551.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  node_features = torch.tensor(node_features, dtype=torch.float)
C:\Users\vladulik\AppData\Local\Temp\ipykernel_3024\1643927551.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_attr = torch.tensor(edge_attr, dtype=torch.float)


In [56]:
class RoutePlannerGNN(nn.Module):
    def __init__(self, node_features, edge_features, hidden_dim):
        super().__init__()
        self.node_encoder = nn.Linear(node_features, hidden_dim)
        self.conv1 = GATConv(hidden_dim, hidden_dim, heads=4, concat=True)
        self.conv2 = GATConv(hidden_dim*4, hidden_dim)
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim*2 + edge_features, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, x, edge_index, edge_attr):
        x = F.relu(self.node_encoder(x))
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        src, dst = edge_index
        edge_repr = torch.cat([x[src], x[dst], edge_attr], dim=1)
        edge_score = self.edge_mlp(edge_repr).squeeze(-1)
        return edge_score

In [62]:
model = RoutePlannerGNN(node_features=3, edge_features=1, hidden_dim=64)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

target_weights = data.edge_attr.squeeze()

for epoch in range(200):
    optimizer.zero_grad()
    pred = model(data.x, data.edge_index, data.edge_attr)
    loss = F.mse_loss(pred, target_weights)
    loss.backward()
    optimizer.step()

print("Loss:", loss.item())


RuntimeError: mat1 and mat2 shapes cannot be multiplied (100x2 and 3x64)